# Code for Estimating eSOH Parameters

In [23]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import gzip
import pickle
import os,sys
import pandas as pd
import scipy
import numpy as np
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
from scipy.optimize import differential_evolution, curve_fit, minimize
from scipy.optimize import Bounds, LinearConstraint, NonlinearConstraint
from scipy import interpolate, integrate
from joblib import Parallel, delayed, parallel_backend
import time

CRED = "\033[41m" #red (terminal color for print statement) 
CEND = "\033[0m" #reset all formatting
plt.rcParams['figure.max_open_warning'] = 0

voltaiq_path = "Z:\\voltaiq_data\\Processed\\GMJuly2022\\GMJuly2022_CELL"
eSOH_export_DIR = "./data/eSOH/"
# %matplotlib qt
%matplotlib widget

## Load Cell Data

In [24]:
def load_RPT_data(cell):
    ## Load RPT pickle
    cell_no = f"{cell:03d}"
    CD_path = voltaiq_path + cell_no + "\\RPT.pkl.gz"

    with gzip.open(CD_path, 'rb') as f:
            CD = pickle.load(f)
    print(cell_no)
    ## Remove Formation data
    idx_Form = (CD.index[CD['Test name'].str.contains('_F_')]).to_list()
    CD = CD.drop(idx_Form,axis=0)
    CD = CD.reset_index(drop=True)
    CD_RPT = CD
    # Remove HPPC data
    idx_HPPC = (CD_RPT.index[CD_RPT["Protocol"]=="HPPC"]).to_list()
    idx_drop = []
    for idx in idx_HPPC:
        prot = CD_RPT["Protocol"].iloc[idx-1]
        td = (CD_RPT["Time [ms]"].iloc[idx]-CD_RPT["Time [ms]"].iloc[idx-1])/1e3/3600
        if td < 10 and prot == "C/20 charge":
                idx_drop.append(idx-1)
    CD_RPTf = CD_RPT.drop(idx_drop,axis=0)
    CD_RPTf = CD_RPTf.drop(idx_HPPC,axis=0)
    CD_RPTf = CD_RPTf.reset_index(drop=True)
    ## Filter Charge and Discharge Data
    Ch_RPT = CD_RPTf[CD_RPTf["Protocol"]=="C/20 charge"]
    Ch_RPT = Ch_RPT.reset_index(drop=True)
    Dh_RPT = CD_RPTf[CD_RPTf["Protocol"]=="C/20 discharge"]
    Dh_RPT = Dh_RPT.reset_index(drop=True)
    Ah_N = Ch_RPT["Ah throughput [A.h]"].to_numpy()

    Ch_RPT = Ch_RPT.reset_index(drop=True)
    Dh_RPT = Dh_RPT.reset_index(drop=True)
    
    return CD,Ch_RPT,Dh_RPT,cell_no

## Process RPT Voltage Data

In [25]:
def filter_data(Qdata,Vdata,window_length=3001,polyorder=3):
    Qf = savgol_filter(Qdata,window_length,polyorder,0)
    dQ = -savgol_filter(Qdata,window_length,polyorder,1)
    Vf = savgol_filter(Vdata,window_length,polyorder,0)
    dV = savgol_filter(Vdata,window_length,polyorder,1)
    dVdQ = dV/dQ
    return Qf,Vf,dVdQ

In [26]:
def load_V_data(i,Ch_RPT,Dh_RPT,d_int=0.01,plot_data_bool=False,plot_int_bool=False):
    d_ch = Ch_RPT["Data"].iloc[i]
    d_dh = Dh_RPT["Data"].iloc[i]
    test_name = Ch_RPT["Test name"].iloc[i]
    AhTh = Ch_RPT["Ah throughput [A.h]"].iloc[i]
    AhTh = round(AhTh,1)
    d_dh = d_dh.reset_index(drop=True)
    d_ch = d_ch.reset_index(drop=True)
    d_dh["Time [ms]"] = d_dh["Time [ms]"]-d_dh["Time [ms]"].iloc[0]
    d_ch["Time [ms]"] = d_ch["Time [ms]"]-d_ch["Time [ms]"].iloc[0]
    d_ch=d_ch[d_ch["Time [ms]"]<=1e8]
    d_dh=d_dh[d_dh["Time [ms]"]<=1e8]
    d_ch1=d_ch[(d_ch["Current [A]"]>0)]
    d_dh1=d_dh[(d_dh["Current [A]"]<0)]
    d_ch=d_ch[(d_ch["Current [A]"]>0.174) & (d_ch["Current [A]"]<0.18)]
    d_dh=d_dh[(d_dh["Current [A]"]<-0.174) & (d_dh["Current [A]"]>-0.18)]
    Q_CV = max(d_ch1["Ah throughput [A.h]"])-max(d_ch["Ah throughput [A.h]"])
    # Filter data points with only V>2.7 V and V< 4.2V
    d_ch=d_ch[(d_ch["Voltage [V]"]>=2.7) & (d_ch["Voltage [V]"]<=4.2)]
    d_dh=d_dh[(d_dh["Voltage [V]"]>=2.7) & (d_dh["Voltage [V]"]<=4.2)]
    d_ch["Ah throughput [A.h]"] = d_ch["Ah throughput [A.h]"]-d_ch["Ah throughput [A.h]"].iloc[0]
    d_dh["Ah throughput [A.h]"] = d_dh["Ah throughput [A.h]"]-d_dh["Ah throughput [A.h]"].iloc[0]

    t_d = d_dh["Time [ms]"].to_numpy()
    t_d = t_d - t_d[0]
    t_d = t_d/1e3 
    I_d = d_dh["Current [A]"].to_numpy()
    V_d = d_dh["Voltage [V]"].to_numpy()
    Ah_d = d_dh["Ah throughput [A.h]"].to_numpy()
    Q_d = integrate.cumtrapz(abs(I_d), t_d/3600)
    Q_d = np.append(Q_d,Q_d[-1])
    t_c = d_ch["Time [ms]"].to_numpy()
    t_c = t_c - t_c[0]
    t_c = t_c/1e3
    I_c = d_ch["Current [A]"].to_numpy()
    V_c = d_ch["Voltage [V]"].to_numpy()
    Ah_c = d_ch["Ah throughput [A.h]"].to_numpy()
    Q_c = integrate.cumtrapz(abs(I_c), t_c/3600)
    Q_c = np.append(Q_c,Q_c[-1])
    ## Normalizing from SOC=100
    Ah_c = Ah_c[-1]-Ah_c + Q_CV
    Q_c = Q_c[-1]-Q_c + Q_CV
    # Interpolate for Averaging
    Q_d,idx_d = np.unique(Q_d,return_index=True)
    V_d = V_d[idx_d]
    Q_c,idx_c = np.unique(Q_c,return_index=True)
    V_c = V_c[idx_c]

    dt = np.average(np.diff(t_d))
    dt = round(dt,1)

    window = int(3000/dt + 1)
    Qf_d,Vf_d,dVdQ_d = filter_data(Q_d,V_d,window_length=window,polyorder=3)
    Qf_c,Vf_c,dVdQ_c = filter_data(Q_c,V_c,window_length=window,polyorder=3)

    Qf_d,idx_d = np.unique(Qf_d,return_index=True)
    Vf_d = Vf_d[idx_d]
    Qf_c,idx_c = np.unique(Qf_c,return_index=True)
    Vf_c = Vf_c[idx_c]
    int_V_d = interpolate.CubicSpline(Qf_d,Vf_d,extrapolate=True)
    int_dVdQ_d = interpolate.CubicSpline(Qf_d,dVdQ_d,extrapolate=True)
    int_V_c = interpolate.CubicSpline(Qf_c,Vf_c,extrapolate=True)
    int_dVdQ_c = interpolate.CubicSpline(Qf_c,dVdQ_c,extrapolate=True)

    Qmax = np.min([np.max(Qf_d),np.max(Qf_c)])
    Qmin = np.max([np.min(Qf_d),np.min(Qf_c)])
    Qfull = np.max([np.max(Qf_d),np.max(Qf_c)])

    Qin = np.arange(Qmin,Qmax,d_int)
    V_d_int = int_V_d(Qin)
    V_c_int = int_V_c(Qin)
    dVdQ_d_int = int_dVdQ_d(Qin)
    dVdQ_c_int = int_dVdQ_c(Qin)
    V_avg = (V_d_int+V_c_int)/2
    dVdQ_avg = (dVdQ_d_int+dVdQ_c_int)/2

    if plot_data_bool:
        fig,ax = plt.subplots(1,1,figsize=(5,4))
        ax.plot(Q_c,V_c,'b')
        ax.plot(Q_d,V_d,'r')
        ax.set_xlabel("Q [Ah]")
        ax.set_ylabel("Voltage [V]")
        ax.legend(["Charge","Discharge"])
        ax.set_title(f"{test_name}")
    if plot_int_bool:
        fig,ax = plt.subplots(1,1,figsize=(5,4))
        ax.plot(Qin,V_c_int,'b')
        ax.plot(Qin,V_d_int,'r')
        ax.plot(Qin,V_avg,'k')
        ax.set_xlabel("Q [Ah]")
        ax.set_ylabel("Voltage [V]")
        ax.legend(["Charge","Discharge","Avg"])

    return Qin,V_avg,dVdQ_avg,Qfull,AhTh
    

## Define Un,Up

In [27]:
def Un(sto):

    var=[-3.54049607669295,0.00708244334002580,0.00774192469266890, 
    4.26893363759502,-0.0164043013936254,-4.05401806281007, 
    0.0426131798578846,-3.19444157210193,0.0503611972394406, 
    0.170261138869476,0.147567301186300,0.0382504766072001, 
    0.519446050169237,1.10619736534131,0.0145120887836752, 
    -0.0816693980616928,-0.0119716740325398,-0.00723858739498425, 
    -0.0877677643234304,0.0238786887373114,0.0452264234816890, 
    -0.00713413913218840];

    p_eq=var[0:8]
    a_eq=var[8:15]
    b_eq = var[15:]
    u_eq2=p_eq[-1]+p_eq[-2]*np.exp((sto-a_eq[-1])/b_eq[-1])
    for i in range(6):
        u_eq2 += p_eq[i]*np.tanh((sto-a_eq[i])/b_eq[i])

    var = [-2.74740857138957,0.00443109156371119,0.0140962302368559, 
        3.10348817994589,-0.0128948359572101,-4.36083705035769, 
        0.0643328570911640,-3.88703879262989,0.0631828141079049, 
        0.213012646348329,0.174731100372283,0.0577291579271751, 
        0.518982409471130,1.21588542781399,0.0150389780167150, 
        -0.0584524226380269,-0.00702394962468186,-0.0342305292048576, 
        -0.0619846453717142,0.0123530018211038,0.0873816814557679, 
        -0.00843321991754559];
    p_eq=var[0:8]
    a_eq=var[8:15]
    b_eq = var[15:]
    u_eq1=p_eq[-1]+p_eq[-2]*np.exp((sto-a_eq[-1])/b_eq[-1])
    for i in range(6):
        u_eq1 += p_eq[i]*np.tanh((sto-a_eq[i])/b_eq[i])

    u_eq = (u_eq1 + u_eq2)/2;

    return u_eq

def Up(sto):

    p1 = -2253.9364
    p2 = 10756.6071
    p3 = -21755.8183
    p4 = 24277.2504
    p5 = -16299.5659
    p6 = 6728.9153
    p7 = -1670.2785
    p8 = 233.2321
    p9 = -18.3223
    p10 = 5.3936

    u_eq = p1*(sto**9) + p2*(sto**8) + p3*(sto**7) + p4*(sto**6) + p5*(sto**5) + p6*(sto**4) + p7*(sto**3) + p8*(sto**2) + p9*sto + p10

    return u_eq

## Peak Finding

In [28]:
def get_peaks(Q,dVdQ):
    Qmax = max(Q)
    pks_,_ = find_peaks(dVdQ,prominence=0.01)
    pks = [pk for pk in pks_ if Q[pk]>=0.1*Qmax and Q[pk]<=0.9*Qmax ]
    Qpeaks = Q[pks]
    if len(Qpeaks) >0:
        Q1 = Qpeaks[0]
        Q2 = Qpeaks[-1]
    else:
        Q1 =  Q[0]
        Q2 =  Q[-1]
    return Q1,Q2

## OCP and Fit Function

In [29]:
def OCP(X,Q):
    ocp = Up(X[3]+Q/X[2])-Un(X[1]-Q/X[0])
    return ocp

def fitfunc(x,Qdata,Vdata,dVdQdata,w1,w2,w3,dVdQ_bool):
  model = np.concatenate([
         [OCP(x,Q)]
        for Q in Qdata
    ]
  )
  Qa, Qb = get_peaks(Qdata,dVdQdata)
  Qmax = np.max(Qdata)
  # Q1 = 0*0.1*Qmax
  Q1 = 0.1*Qmax
  Q2 = 0.9*Qmax
  if Qa>0 and Qa<0.5*Qmax  and dVdQ_bool:
    Q3 = Qa - 0.05*Qmax
    Q4 = Qa + 0.05*Qmax
  else:
    Q3 = 0.25*Qmax
    Q4 = 0.45*Qmax
  if Qb< Qmax and Qb > 0.5*Qmax and dVdQ_bool:
    Q5 = Qb - 0.05*Qmax
    Q6 = Qb + 0.05*Qmax
  else:
    Q5 = 0.7*Qmax
    Q6 = 0.9*Qmax
  wvec = np.ones(len(Qdata))
  for i,Q in enumerate(Qdata):
    if Q<Q1 or Q>Q2:
        wvec[i]=w1
    if Q>=Q1 and Q<=Q2:
        wvec[i]=w2
    if Q>=Q3 and Q<=Q4:
        wvec[i]=w3
    if Q>=Q5 and Q<=Q6:
        wvec[i]=w3    
  Vd = np.multiply(Vdata,wvec)
  Vs = np.multiply(model,wvec)
  error = Vd-Vs
  out = np.linalg.norm(error)/np.sqrt(len(Vd))
  return out

## RPT eSOH fitting algorithm

In [30]:
def esoh_est(i,Qdata,Vdata,dVdQ_data,Qfull,AhTh,w1=1,w2=1,w3=1,dVdQ_bool=True,x0 = [4.2,0.85,5.5,0.3], lb = [1, 0, 1, 0],ub = [5, 1, 6.5, 1],plot_bool=False,win=5,version=6):
    Cap = Qfull
    bounds = Bounds(lb, ub)
    nleq1 = lambda x: -Cap/x[0]+x[1]
    nlcon1 = NonlinearConstraint(nleq1, 0, 1)
    nleq2 = lambda x:  Cap/x[2]+x[3]
    nlcon2 = NonlinearConstraint(nleq2, 0, 1)
    if i==0:
        result = minimize(fitfunc, x0, args=(Qdata,Vdata,dVdQ_data,w1,w2,w3,dVdQ_bool), bounds=bounds,constraints=[nlcon1,nlcon2])
    else:
        result = minimize(fitfunc, x0, args=(Qdata,Vdata,dVdQ_data,w1,w2,w3,dVdQ_bool), bounds=bounds,constraints=[nlcon1,nlcon2])
    res = result.x
    Cn = res[0]
    Cp = res[2]
    x100 = res[1]
    y100 = res[3]
    x0 = res[1]-Cap/res[0]
    y0 = res[3]+Cap/res[2]
    theta = [Cn,x0,x100,Cp,y0,y100]
    theta = [round(tt,4) for tt in theta]
    Vfit = np.concatenate([
            [OCP(res,Q)]
            for Q in Qdata
        ]
    )
    err_V = 1e3*np.linalg.norm(Vdata-Vfit)/np.sqrt(len(Vdata))
    err_V = round(err_V,1)
    Cap = round(Cap,3)
    _,_,dVdQ_fit = filter_data(Qdata,Vfit,window_length=win,polyorder=3)
    Q1_data,Q2_data = get_peaks(Qdata,dVdQ_data)
    Q1_fit,Q2_fit = get_peaks(Qdata,dVdQ_fit)
    p1_err = Q1_data-Q1_fit
    p2_err = Q2_data-Q2_fit
    p12_data = Q2_data-Q1_data
    p12_fit = Q2_fit-Q1_fit
    p12_err = p12_data-p12_fit
    Q_bool = (Qdata>0.1*Cap) & (Qdata<0.9*Cap)
    dVdQ_data_f = dVdQ_data[Q_bool]
    dVdQ_fit_f = dVdQ_fit[Q_bool]
    Q_f = Qdata[Q_bool]
    
    err_dVdQ = 1e3*np.linalg.norm(dVdQ_data_f-dVdQ_fit_f)/np.sqrt(len(Vdata))
    err_dVdQ = round(err_dVdQ,1)
    if plot_bool:
        fig,ax = plt.subplots(1,2,figsize=(10,4))
        ax1 = ax.flat[0]
        ax1.plot(Qdata,Vdata,'k')
        ax1.plot(Qdata,Vfit,'r--')
        ax1.set_xlabel("Q [Ah]")
        ax1.set_ylabel("Voltage [V]")
        ax1.legend(["Data","Fit"])
        ax1.set_title(f"RPT # {i+1}, Ah_th={AhTh}")
        
        ver = int(str(version)[0])
        if ver != 6:
            if ver == 7:
                Qarr = [0.1*Cap,0.9*Cap]
            else:
                if ver == 8:
                    Qarr = [0.1*Cap,0.25*Cap,0.45*Cap,0.7*Cap,0.9*Cap,0.9*Cap]
                elif ver == 9:
                    Qarr = [0.1*Cap,Q1_data-0.05*Cap,Q1_data+0.05*Cap,
                            Q2_data-0.05*Cap,Q2_data+0.05*Cap,0.9*Cap]
                ax1.axvspan(Qarr[1],Qarr[2], alpha=0.1, color='green')
                ax1.axvspan(Qarr[3],Qarr[4], alpha=0.1, color='green')
            # ax1.vlines(Qarr,2.8,4.2,colors='k',linestyles='--')
            ax1.axvspan(0, Qarr[0], alpha=0.1, color='red')
            ax1.axvspan(Qarr[-1],Cap, alpha=0.1, color='red')
        ax1.text(0.5,0.2,f'RMSE: {err_V:0.1f} mV',transform=ax1.transAxes)
        ax2 = ax.flat[1]
        ax2.plot(Q_f,dVdQ_data_f,'k')
        ax2.plot(Q_f,dVdQ_fit_f,'r--')
        ax2.vlines([Q1_data,Q2_data],0,0.6,colors="k",linestyles="--")
        ax2.vlines([Q1_fit,Q2_fit],0,0.6,colors="r",linestyles=":")
        ax2.set_xlabel("Q [Ah]")
        ax2.set_ylabel("dVdQ")
        ax2.legend(["Data","Fit"])
        # ax2.set_ylim([0,0.6])
        fig.tight_layout()
    else:
        fig = []
    return theta,Cap,err_V,err_dVdQ,p1_err,p2_err,p12_err,fig

## Cell eSOH Estimation Code

In [31]:
def run_esoh_est(cell,d_int=0.01,w1=0.2,w2=1,w3=2,plot_data_bool=False,plot_int_bool=False,plot_fit_bool=False,debug_N=0,dVdQ_bool=False,version=86):
    try:
        CD,Ch_RPT,Dh_RPT,cell_no = load_RPT_data(cell)
        if debug_N == 0:
            N_RPT = len(Ch_RPT)
        else:
            N_RPT = debug_N
        theta = np.zeros([N_RPT,6])
        AhTh = np.zeros(N_RPT)
        Cap = np.zeros(N_RPT)
        err_V = np.zeros(N_RPT)
        err_dVdQ = np.zeros(N_RPT)
        p1_err = np.zeros(N_RPT)
        p2_err = np.zeros(N_RPT)
        p12_err = np.zeros(N_RPT)
        for i in range(N_RPT):
            try:
                # print(i)
                Qdata, Vdata,dVdQdata,Qfull,AhTh[i] = load_V_data(i,Ch_RPT,Dh_RPT,d_int=d_int,plot_data_bool=plot_data_bool,plot_int_bool=plot_int_bool)
                theta[i],Cap[i],err_V[i],err_dVdQ[i],p1_err[i],p2_err[i],p12_err[i],fig= esoh_est(i,Qdata,Vdata,dVdQdata,Qfull,AhTh[i],w1=w1,w2=w2,w3=w3,dVdQ_bool=dVdQ_bool,plot_bool=plot_fit_bool,version=version)
                if err_V[i]>20:
                    print("High RMSE")
                    print(Ch_RPT["Test name"].iloc[i])
                    theta[i,:]=np.NaN
            except:
                theta[i] = np.NaN
                Cap[i] = np.NaN
                err_V[i] = np.NaN
                err_dVdQ[i] = np.NaN
                p1_err[i] = np.NaN
                p2_err[i] = np.NaN
                p12_err[i] = np.NaN
                print(f"Error in processing RPT {i+1}")
                print(Ch_RPT["Test name"].iloc[i])
        Cn = []; x0 = []; x100 = []
        Cp = []; y0 = []; y100 = []
        for i in range(len(theta)):
            Cn.append(theta[i][0])
            x0.append(theta[i][1])
            x100.append(theta[i][2])
            Cp.append(theta[i][3])
            y0.append(theta[i][4])
            y100.append(theta[i][5])
        df = pd.DataFrame({'AhTh':AhTh,'C':Cap,'Cn':Cn,'x0':x0,'x100':x100,'Cp':Cp,'y0':y0,'y100':y100,
            'RMSE_V':err_V,'RMSE_dVdQ':err_dVdQ,'p1_err':p1_err,'p2_err':p2_err,'p12_err':p12_err})
        if debug_N == 0:
            df.to_csv(eSOH_export_DIR+f"GMJuly2022_CELL"+cell_no+f"_eSOH.csv", index=False)
        else:
            df.to_csv(eSOH_export_DIR+f"GMJuly2022_CELL"+cell_no+"_eSOH_debug.csv", index=False)
    except Exception as e:
        stream = getattr(sys, "stdout")
        print(CRED+f"Error: {cell:03d}"+CEND, file=stream)
        print(e, file=stream) 
        stream.flush()
        df = []

# Single Cell

In [32]:
cell = 58
run_esoh_est(cell)

058


# Parallel For Loop

In [ ]:
cells = range(120)
with parallel_backend("loky",n_jobs=-1):#"loky"
    yyy1 = Parallel()(delayed(run_esoh_est)(cell) for cell in cells)

# For Loop

In [ ]:
cells = range(120)
for cell in cells:
    run_esoh_est(cell)